[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_10/02_tag_10_diffusion_unet_embeddings_lokal.ipynb)

# Tag 10 – 02: Diffusion, U-Net und Text-Embeddings – lokal Bilder erzeugen

Dieses Notebook führt ein vortrainiertes **Stable Diffusion 1.5** lokal aus. Das Modell besteht aus einem Text-Encoder, einem U-Net zur Rauschvorhersage und einem VAE zum Umwandeln zwischen Bild- und Latentraum. Es wird beim ersten Lauf von Hugging Face heruntergeladen und anschließend aus einem lokalen Ordner benutzt.

## Von GAN zu Diffusion

| Ansatz | Wie entsteht ein Bild? | Typische Stärke |
| --- | --- | --- |
| GAN | Generator täuscht einen Diskriminator in einem Gegenspiel. | Sehr schnelle Erzeugung nach dem Training. |
| Diffusion | Rauschen wird über viele kleine Schritte in ein Bild zurückgeführt. | Stabileres Training, sehr gute Prompt-Steuerung. |

Beim Training wird zu einem echten Bild $x_0$ künstliches Rauschen addiert: $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$. Das U-Net erhält $x_t$, den Zeitschritt $t$ und den Textkontext und lernt, das Rauschen $\epsilon$ vorherzusagen. Bei der Erzeugung beginnt man mit Zufallsrauschen und läuft diese Schritte rückwärts.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from diffusers import StableDiffusionPipeline
from huggingface_hub import snapshot_download

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Dieses Notebook ist für lokale GPU-Inferenz gedacht. Bitte CUDA-PyTorch installieren.')

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print('CUDA verfügbar:', torch.cuda.is_available())

# Gute Startwerte für eine RTX 3080 (10 GB): 512×512, Batch 1 und float16.
WIDTH = HEIGHT = 512
STEPS = 30
GUIDANCE_SCALE = 7.5
SEED = 42

## Die Modellstruktur: von Text zu Bild

Stable Diffusion rechnet nicht direkt auf einem 512×512×3-Pixelbild, sondern in einem **Latentraum**. Für ein Bild dieser Größe hat ein Latent bei SD 1.5 die Form `(Batch, 4, 64, 64)`: achtmal weniger Höhe und Breite, vier statt drei Kanäle. Das spart ungefähr Faktor 64 an räumlichen Rechenoperationen gegenüber einem U-Net auf Pixeln.

| Baustein | Einfach gesagt | Input → Output | Wird beim ursprünglichen SD-Training gelernt? |
| --- | --- | --- | --- |
| VAE-Encoder | komprimiert ein Bild in eine kleine Bildbeschreibung | `RGB-Bild` → `Latent z₀` | bereits vortrainiert und später eingefroren |
| CLIP-Text-Encoder | übersetzt Prompt-Tokens in Zahlenvektoren mit Bedeutung | `Text` → `(1, 77, 768)` Embeddings | bereits vortrainiert und eingefroren |
| Diffusions-U-Net | schätzt, welches Rauschen im Latent steckt | `zₜ`, Zeitpunkt `t`, Textkontext → `ε_θ` | **ja: das ist der zentrale trainierte Teil** |
| VAE-Decoder | macht aus dem finalen Latent wieder Pixel | `Latent z₀` → `RGB-Bild` | bereits vortrainiert und eingefroren |

Bei der Bildgenerierung in diesem Notebook wird nur noch der **Text-Encoder, U-Net und VAE-Decoder ausgeführt**. Der VAE-Encoder ist für das Training oder für Bild-zu-Bild-Aufgaben nötig, nicht für reines Text-zu-Bild.

## Hugging Face: einmalig herunterladen

Der Download und die Anmeldung sind für dieses öffentliche Modell **nicht nötig**, solange er bei dir ohne Fehler startet. Ein Hugging-Face-Login bzw. `HF_TOKEN` wird erst bei privaten oder zugriffsbeschränkten Modellen relevant.

Die nächste Zelle lädt nur dann, wenn der Ordner `saved_models/stable-diffusion-v1-5/` noch nicht existiert. Existiert er, wird sein Inhalt direkt verwendet – ohne Download-Prüfung und ohne Netzwerkanfrage. Der Modellordner (rund 4–5 GB) sowie erzeugte Bilder sind durch `.gitignore` ausgeschlossen.

**Hinweis:** Ein lediglich angelegter, aber unvollständiger Ordner zählt dabei ebenfalls als vorhanden. In diesem seltenen Fall den Ordner löschen und die Zelle noch einmal ausführen.

In [ ]:
MODEL_ID = 'runwayml/stable-diffusion-v1-5'
# Funktioniert sowohl bei Jupyter im Projektwurzelordner als auch bei direkt geöffnetem Notebook.
COURSE_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                    if (folder / 'requirements.txt').exists()), Path.cwd())
DAY_DIR = COURSE_ROOT / 'notebooks' / 'Day_10'
MODEL_DIR = DAY_DIR / 'saved_models' / 'stable-diffusion-v1-5'
# Optional: nur für private bzw. zugriffsbeschränkte Modelle nötig.
HF_TOKEN = os.environ.get('HF_TOKEN')

if MODEL_DIR.exists():
    # Kein snapshot_download: vorhandener Ordner bedeutet bewusst kein Netzwerkkontakt.
    model_path = str(MODEL_DIR)
    print('Vorhandenes lokales Modell wird verwendet – Download übersprungen.')
else:
    try:
        model_path = snapshot_download(
            repo_id=MODEL_ID,
            local_dir=MODEL_DIR,
            token=HF_TOKEN,
        )
        print('Modell einmalig heruntergeladen.')
    except Exception as error:
        raise RuntimeError(
            'Download fehlgeschlagen. Bei einem eingeschränkten Modell helfen Login oder ein HF_TOKEN.'
        ) from error

print('Lokaler Modellpfad:', model_path)

In [ ]:
# float16 halbiert den Speicherbedarf gegenüber float32.
pipe = StableDiffusionPipeline.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    local_files_only=True,  # Der vorherige Schritt liefert immer einen lokalen Pfad.
)

# Die RTX 3080 kann SD 1.5 bei 512×512 normalerweise vollständig im VRAM halten.
# Bei Out-of-Memory: diese zwei Zeilen statt pipe.to(DEVICE) verwenden.
USE_CPU_OFFLOAD = False
if USE_CPU_OFFLOAD:
    pipe.enable_model_cpu_offload()  # spart VRAM, ist aber langsamer
else:
    pipe.to(DEVICE)

print('U-Net-Eingangskanäle:', pipe.unet.config.in_channels)
print('U-Net-Textkontext-Dimension:', pipe.unet.config.cross_attention_dim)
print('U-Net-Encoder-Blöcke:', list(pipe.unet.config.down_block_types))
print('U-Net-Decoder-Blöcke:', list(pipe.unet.config.up_block_types))
print('Kanäle je Auflösung:', list(pipe.unet.config.block_out_channels))
print('VAE-Skalierungsfaktor:', pipe.vae_scale_factor)
print('Latent-Auflösung für 512×512:', HEIGHT // pipe.vae_scale_factor, '×', WIDTH // pipe.vae_scale_factor)

## Was wird beim Diffusionstraining minimiert?

Für ein Trainingsbild wird zunächst der VAE-Encoder einmalig zu einem sauberen Latent $z_0$ komprimiert. Dann wählt man zufällig einen Zeitschritt $t$ und zufälliges gaußsches Rauschen $\epsilon\sim\mathcal{N}(0,I)$. Der **Scheduler** mischt dieses bekannte Rauschen in $z_0$ und liefert $z_t$. Der Scheduler hat keine lernbaren Parameter.

Das U-Net sieht `zₜ`, den Zeitschritt und die Text-Embeddings $c$. Seine Ausgabe $\epsilon_\theta(z_t,t,c)$ soll exakt das Rauschen treffen, das wir selbst hineingemischt haben. Minimiert wird der mittlere quadratische Fehler:

$$L_{simple}=\mathbb{E}_{z_0,\epsilon,t,c}\left[\lVert\epsilon-\epsilon_\theta(z_t,t,c)\rVert_2^2\right].$$

Es gibt also kein fertiges Zielbild, das der U-Net direkt ausgeben soll: Sein Target ist das zufällig bekannte **Rauschen**. Kann es das gut vorhersagen, kann der Scheduler einen kleinen Rauschanteil abziehen. Viele solcher Rückwärtsschritte führen von reinem Zufall zu einem Bildlatent.

## U-Net einfach zerlegt: Encoder → Mitte → Decoder

```text
verrauschtes Latent zₜ: (4, 64, 64)
          │  + Zeitschritt-Embedding + Text-Embeddings
          ▼
Encoder / Down-Pfad: 64² → 32² → 16² → 8²
  ResNet-Blöcke erkennen erst lokale Muster, dann immer größeren Kontext.
  Die Feature-Kanäle nehmen dabei zu; jede Stufe wird als Skip gespeichert.
          ▼
Bottleneck / Mitte: kleinste Karte, größter Bildkontext
          ▼
Decoder / Up-Pfad: 8² → 16² → 32² → 64²
  Hochskalieren + passende Skip-Features aus dem Encoder zusammenführen.
          ▼
vorhergesagtes Rauschen ε_θ: (4, 64, 64)
```

**Warum Skip Connections?** Beim Downsampling gehen genaue Positionen und Kanten verloren. Der Decoder bekommt deshalb die hochaufgelösten Zwischenfeatures direkt aus derselben Encoder-Stufe. Die Mitte liefert das *Was* (globaler Kontext), die Skips das *Wo* (feine Lage und Kanten).

**Wie kommt Text hinein?** Aus den räumlichen U-Net-Features entstehen in Cross-Attention die Queries $Q$; aus den CLIP-Embeddings entstehen Keys $K$ und Values $V$. Vereinfacht: Jeder Bildbereich kann dabei lernen, auf die zu ihm passenden Wörter zu achten. Zeit-Embeddings werden zusätzlich in die ResNet-Blöcke eingebracht, damit das Netz erkennt, ob es grobes oder schon sehr feines Rauschen entfernen soll.

Bei der Erzeugung wird derselbe U-Net zweimal abgefragt: einmal ohne Prompt und einmal mit Prompt. Classifier-free guidance kombiniert beide Vorhersagen zu $\epsilon_{guided}=\epsilon_{uncond}+s(\epsilon_{text}-\epsilon_{uncond})$. $s$ ist hier `GUIDANCE_SCALE`; ein größerer Wert folgt dem Text stärker, kann aber Artefakte erzeugen.

## Prompt → Tokens → Embeddings

Der String selbst wird nicht in das U-Net gegeben. CLIP tokenisiert den Prompt und bildet für jeden Token einen Vektor. Das Tensorformat ist normalerweise `(Batch, Token, Embedding-Dimension)`. Wir berechnen die Embeddings explizit und geben sie danach an die Pipeline weiter – so wird der Zusammenhang sichtbar.

In [ ]:
prompt = 'a small red fox reading a book in a cozy library, warm light, detailed illustration'
negative_prompt = 'blurry, low quality, distorted, text, watermark'

tokenized = pipe.tokenizer(
    prompt,
    padding='max_length',
    max_length=pipe.tokenizer.model_max_length,
    truncation=True,
    return_tensors='pt',
)
print('Token-IDs:', tuple(tokenized.input_ids.shape))
print('Erste Token:', pipe.tokenizer.convert_ids_to_tokens(tokenized.input_ids[0][:12]))

execution_device = pipe._execution_device
with torch.inference_mode():
    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=prompt,
        negative_prompt=negative_prompt,
        device=execution_device,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
    )

print('Prompt-Embeddings:', tuple(prompt_embeds.shape), prompt_embeds.dtype)
print('Negative-Embeddings:', tuple(negative_prompt_embeds.shape), negative_prompt_embeds.dtype)

## Lokal erzeugen

Der Seed macht den anfänglichen Rauschvektor reproduzierbar. Bei gleichem Modell, Prompt, Parametern und Seed entsteht daher sehr ähnlich dasselbe Bild. Mehr `STEPS` verbessert nicht automatisch jedes Bild; 25–40 ist ein sinnvoller Bereich für den Einstieg.

In [ ]:
generator = torch.Generator(device=execution_device).manual_seed(SEED)

with torch.inference_mode():
    result = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )

image = result.images[0]
output_dir = DAY_DIR / 'generated_images'
output_dir.mkdir(exist_ok=True)
output_path = output_dir / f'fox_seed_{SEED}.png'
image.save(output_path)

plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.title(f'Seed {SEED} | {STEPS} Schritte | Guidance {GUIDANCE_SCALE}')
plt.axis('off')
plt.show()
print('Gespeichert:', output_path.resolve())

## Experimentieren

- Ändere zuerst nur den `SEED`: Komposition und Motivvarianten ändern sich, die Prompt-Bedeutung bleibt erhalten.
- Halte den Seed fest und verändere einzelne Promptteile, etwa Stil oder Licht.
- Probiere `GUIDANCE_SCALE` zwischen 5 und 10. Zu hoch kann unnatürliche oder überzeichnete Bilder erzeugen.
- Bleibe auf der RTX 3080 zunächst bei 512×512 und einem Bild gleichzeitig. Bei Speicherfehlern `USE_CPU_OFFLOAD = True` setzen; für große Auflösungen lieber anschließend hochskalieren.

## Zusammenfassung

- Diffusion lernt das Umkehren eines definierten Rauschprozesses.
- Das U-Net sagt in jedem Schritt das Rauschen voraus; Skip Connections bewahren räumliche Details.
- Text-Embeddings konditionieren das U-Net über Cross-Attention und machen Text-zu-Bild möglich.
- Hugging Face lädt Modellgewichte lokal. Nach dem ersten Download kann dieselbe Pipeline ohne Cloud-Inferenz und ohne erneuten Download laufen.